# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook guides you through loading, exploring, and analyzing the [FAIR<sup>2</sup> mass spectrometry dataset](https://sen.science/doi/10.71728/senscience.vq4a-28xa) using the [`mlcroissant`](https://mlcommons.org/croissant/) library.

### Dataset Source
The dataset is described using the Croissant schema standard and is accessible via the following URL.

In [ ]:
# Ensure `mlcroissant` is installed
!pip install mlcroissant --quiet

## 1. Data Loading
Load the dataset metadata and record structure from the Croissant schema using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import pprint

# Dataset Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json"

# Load dataset
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata  # This is an MLCroissant object

print(f"Dataset name: {metadata.name}")
print(f"Version: {metadata.version}")
print(f"Description: {metadata.description}\n")

## 2. Data Overview
Examine available record sets, the identifiers (`@id`) for each, and the fields within. All entities are referenced by their Croissant `@id`.

In [ ]:
# List all record sets and their details
def print_recordsets(metadata):
    print("Available record sets:\n")
    record_sets = getattr(metadata, 'record_sets', []) or getattr(metadata, 'recordSet', [])
    if not record_sets:
        print("No record sets found in this Croissant metadata.\n")
        print("Attempting to enumerate DataDistributions (data files) instead.\n")
        if hasattr(metadata, 'distribution'):
            for ds in metadata.distribution:
                print(f"Distribution: id={ds['@id']}")
                if 'name' in ds:
                    print(f"  Name: {ds['name']}")
                if 'description' in ds:
                    print(f"  Description: {ds['description']}")
        return []

    for recset in record_sets:
        print(f"RecordSet: {recset['@id']}")
        print(f"  Name: {recset.get('name', 'N/A')}")
        field_ids = [f['@id'] for f in recset.get('field', [])]
        print(f"  Fields: {field_ids}\n")
    return [rs['@id'] for rs in record_sets]

# Most Croissant schemas list record sets in the 'recordSet' field
record_sets = getattr(metadata, 'recordSet', [])
if not record_sets:
    print("No record sets listed in schema metadata. Will try to inspect records directly.\n")
else:
    for i, record_set in enumerate(record_sets):
        print(f"[{i}] RecordSet: {record_set['@id']}")
        print(f"    Name: {record_set.get('name', 'N/A')}")
        columns = [f["@id"] for f in record_set.get('field', [])]
        print(f"    Fields: {columns}\n")

# If record sets are not present (as some Croissant schemas inline a single file), list available objects for guidance.
if not record_sets:
    print("Croissant file does not explicitly list record sets. Proceeding to inspect available records...")

## 3. Data Extraction
Because this dataset doesn't use Croissant `recordSet` structures, we'll use the `records()` method without specifying a record set, which yields the main tabular data. We'll explore a small sample and display field names (columns).

In [ ]:
# Load main tabular data (records)
row_samples = []
for i, rec in enumerate(dataset.records()):
    row_samples.append(rec)
    if i >= 4:
        break
print("Sample records (first 5):")
pprint.pprint(row_samples)

# Load ALL records for main analysis dataframe
all_records = list(dataset.records())
df = pd.DataFrame(all_records)
print(f"\nColumns found: {list(df.columns)}\n")
df.head()

## 4. Exploratory Data Analysis (EDA)
Let's process and explore the data. We'll find a numeric field (e.g., `MW [kDa]`, likely representing molecular weights), filter proteins above a certain threshold, normalize that field, and optionally group by a categorical field such as `Description` or another identifier. 

All references are by their column name (directly from Croissant's field `@id` where possible).

In [ ]:
# Typical numeric analysis field: MW [kDa] (Molecular Weight in kilodaltons)
numeric_field = 'MW [kDa]'
if numeric_field not in df.columns:
    print(f"Numeric field '{numeric_field}' not found. Available fields:\n{df.columns}")

# Clean up non-numeric entries if present
df[numeric_field] = pd.to_numeric(df[numeric_field], errors='coerce')

# Remove records with missing (NaN) MW values
filtered_df = df[df[numeric_field].notna() & (df[numeric_field] > 10)]
print(f"Records with {numeric_field} > 10 kDa: {filtered_df.shape[0]}")
print(filtered_df[[numeric_field]].head())

# Normalize MW
filtered_df[f"{numeric_field}_normalized"] = (
    filtered_df[numeric_field] - filtered_df[numeric_field].mean()
) / filtered_df[numeric_field].std()
print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

# Group by a descriptive field if it exists
group_field = 'Description' if 'Description' in filtered_df.columns else filtered_df.columns[0]
grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
print(f"\nMean {numeric_field} by {group_field} (first 5 groups):")
print(grouped_df.head())

## 5. Visualization
Let's visualize the distribution of protein molecular weights using a histogram, and plot normalized MW for a quick overview. You can adapt field references by their names as they appear in the DataFrame.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(8,4))
sns.histplot(filtered_df[numeric_field].dropna(), bins=30, kde=True, color='dodgerblue')
plt.title(f"Distribution of {numeric_field}")
plt.xlabel(f"{numeric_field} (kDa)")
plt.ylabel("Count")
plt.show()

# Boxplot of normalized MW for visual outlier detection
plt.figure(figsize=(8,4))
sns.boxplot(x=filtered_df[f"{numeric_field}_normalized"], color='orange')
plt.title(f"Boxplot of Normalized {numeric_field}")
plt.xlabel(f"Normalized {numeric_field}")
plt.show()

## 6. Conclusion

- The dataset, as described by the Croissant schema, contains rich mass spectrometry results with fields including protein accession, description, molecular weight, and more.
- After data extraction with `mlcroissant`, we performed filtering and normalization on molecular weights and visualized their distribution.
- The Croissant `@id` referencing enables robust and transparent dataset interaction for downstream analysis and reproducibility.

You can further explore other fields (columns), aggregate by other biological dimensions, or merge with additional datasets as described by Croissant schemas.